In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# =========================
# 1. Imports
# =========================
import os, json
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV

# =========================
# 2. LOAD DATA
# =========================
base_path = "/kaggle/input/datasets/nikunjnawal009/randomf-primevul"

def load_jsonl(file):
    data = []
    with open(file, "r") as f:
        for line in f:
            obj = json.loads(line)

            code = obj.get("func_before") or obj.get("code") or obj.get("func") or ""
            label = obj.get("target", obj.get("label", 0))

            if code.strip():
                data.append((code, int(label)))

    return data

train_data = load_jsonl(os.path.join(base_path, "primevul_train_paired.jsonl"))
val_data   = load_jsonl(os.path.join(base_path, "primevul_valid_paired.jsonl"))
test_data  = load_jsonl(os.path.join(base_path, "primevul_test_paired.jsonl"))

train_data += val_data

X_train = [x[0] for x in train_data]
y_train = [x[1] for x in train_data]
X_test  = [x[0] for x in test_data]
y_test  = [x[1] for x in test_data]

print("Train size:", len(X_train))
print("Test size:", len(X_test))

# =========================
# 3. TF-IDF (STRONG VERSION)
# =========================
vectorizer = TfidfVectorizer(
    max_features=25000,
    ngram_range=(1,2),
    min_df=3,
    max_df=0.9,
    token_pattern=r'\b\w+\b'
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

# =========================
# 4. RANDOM FOREST (TUNED)
# =========================
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    max_features="sqrt",
    class_weight="balanced_subsample",   # more stable than "balanced"
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train_vec, y_train)

# =========================
# 5. SMART THRESHOLD TUNING
# =========================
probs = rf.predict_proba(X_test_vec)[:, 1]

best_f1 = 0
best_t = 0.5
best_preds = None

print("\n🔍 Threshold tuning:")

for t in np.arange(0.4, 0.7, 0.05):
    preds = (probs > t).astype(int)
    report_temp = classification_report(y_test, preds, output_dict=True)

    recall_0 = report_temp['0']['recall']
    recall_1 = report_temp['1']['recall']
    f1 = report_temp['1']['f1-score']

    print(f"t={t:.2f} → F1={f1:.4f}, R0={recall_0:.2f}, R1={recall_1:.2f}")

    # Avoid broken models
    if recall_0 > 0.3 and recall_1 > 0.3:
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
            best_preds = preds

print("\nBest threshold:", best_t)

# =========================
# 6. FINAL EVALUATION
# =========================
final_preds = best_preds

acc = accuracy_score(y_test, final_preds)
report = classification_report(y_test, final_preds, output_dict=True)

print("\n✅ Accuracy:", acc)
print("\n📊 Classification Report:\n", classification_report(y_test, final_preds))

# =========================
# 7. SAVE RESULTS
# =========================
label_key = [k for k in report.keys() if k.startswith("1")][0]

results = {
    "Model": "RandomForest_Stable",
    "Dataset": "PrimeVul",
    "Accuracy": acc,
    "Precision_vuln": report[label_key]['precision'],
    "Recall_vuln": report[label_key]['recall'],
    "F1_vuln": report[label_key]['f1-score'],
    "Macro_F1": report['macro avg']['f1-score'],
    "Best_threshold": best_t
}

pd.DataFrame([results]).to_csv("/kaggle/working/rf_primevul_final.csv", index=False)

print("\n✅ Final results saved")

Train size: 8538
Test size: 870

🔍 Threshold tuning:
t=0.40 → F1=0.6662, R0=0.00, R1=1.00
t=0.45 → F1=0.6598, R0=0.07, R1=0.95
t=0.50 → F1=0.4801, R0=0.64, R1=0.43
t=0.55 → F1=0.0046, R0=1.00, R1=0.00
t=0.60 → F1=0.0000, R0=1.00, R1=0.00
t=0.65 → F1=0.0000, R0=1.00, R1=0.00

Best threshold: 0.5

✅ Accuracy: 0.5344827586206896

📊 Classification Report:
               precision    recall  f1-score   support

           0       0.53      0.64      0.58       435
           1       0.54      0.43      0.48       435

    accuracy                           0.53       870
   macro avg       0.54      0.53      0.53       870
weighted avg       0.54      0.53      0.53       870


✅ Final results saved


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m